In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import os


base_path = '/content/drive/MyDrive/DataStorm_Project/data'
sub_folders = ['bronze', 'silver', 'gold', 'rejected']

print("Folder Creating...")

for folder in sub_folders:
    folder_path = os.path.join(base_path, folder)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Folder create success: {folder_path}")
    else:
        print(f"Have folder {folder_path}")

print("\n All Folders create success ")

Folder Creating...
Have folder /content/drive/MyDrive/DataStorm_Project/data/bronze
Have folder /content/drive/MyDrive/DataStorm_Project/data/silver
Have folder /content/drive/MyDrive/DataStorm_Project/data/gold
Have folder /content/drive/MyDrive/DataStorm_Project/data/rejected

 All Folders create success 


In [3]:
bronze_path = '/content/drive/MyDrive/DataStorm_Project/data/bronze/'

In [4]:
silver_path = '/content/drive/MyDrive/DataStorm_Project/data/silver/'

In [5]:
rejected_path = '/content/drive/MyDrive/DataStorm_Project/data/rejected/'

In [6]:
# 1. Reusable Function that checks for duplicates

def check_duplicates(df, subset_cols, dataset_name):
    duplicates = df[df.duplicated(subset=subset_cols, keep=False)]
    clean_df = df.drop_duplicates(subset=subset_cols, keep='first')
    if not duplicates.empty:
        duplicates.to_csv(f'{rejected_path}{dataset_name}_duplicates.csv', index=False)
        print(f"  [-] {dataset_name}: Amount of duplicates (with errors) removed -> {len(duplicates)}")
    return clean_df

In [7]:
# 2. Reusable Function that checks for Nulls (empty spaces)

def check_nulls(df, mandatory_cols, dataset_name):
    null_mask = df[mandatory_cols].isnull().any(axis=1)
    null_records = df[null_mask]
    clean_df = df[~null_mask]
    if not null_records.empty:
        null_records.to_csv(f'{rejected_path}{dataset_name}_nulls.csv', index=False)
        print(f"  [-] {dataset_name}:Null (empty) records removed -> {len(null_records)}")
    return clean_df

In [8]:
# 3. Reusable Function that Checks for Negative Values
def check_negative_values(df, numeric_cols, dataset_name):
    mask = (df[numeric_cols] < 0).any(axis=1)
    rejected_records = df[mask]
    clean_df = df[~mask]
    if not rejected_records.empty:
        rejected_records.to_csv(f'{rejected_path}{dataset_name}_negatives.csv', index=False)
        print(f"  [-] {dataset_name}: Number of negative records removed -> {len(rejected_records)}")
    return clean_df

print("Data cleaning begins....\n")

Data cleaning begins....



In [9]:
# --- Outlet Master Data Cleanup ---
print("1. Cleaning Outlet Master data...")
df_outlet = pd.read_csv(bronze_path + 'outlet_master.csv')
df_outlet_clean = check_duplicates(df_outlet, ['Outlet_ID'], 'outlet_master')
df_outlet_clean = check_nulls(df_outlet_clean, ['Outlet_ID', 'Outlet_Type'], 'outlet_master')
df_outlet_clean.to_csv(silver_path + 'clean_outlet_master.csv', index=False)

1. Cleaning Outlet Master data...


In [10]:
# --- Clearing Transactions History Data ---
print("\n2. Clearing Transactions History data ...")
df_trans = pd.read_csv(bronze_path + 'transactions_history_final.csv')
df_trans_clean = check_duplicates(df_trans, df_trans.columns, 'transactions')
df_trans_clean = check_nulls(df_trans_clean, ['Outlet_ID', 'Volume_Liters', 'Total_Bill_Value'], 'transactions')
df_trans_clean = check_negative_values(df_trans_clean, ['Volume_Liters', 'Total_Bill_Value'], 'transactions')
df_trans_clean.to_csv(silver_path + 'clean_transactions.csv', index=False)


2. Clearing Transactions History data ...
  [-] transactions: Number of negative records removed -> 4753


In [11]:
print("\n All data was successfully cleaned and stored in the Silver Layer! ")


 All data was successfully cleaned and stored in the Silver Layer! 


In [11]:
import numpy as np

In [10]:
import pandas as pd

In [14]:
silver_path = '/content/drive/MyDrive/DataStorm_Project/data/silver/'

In [15]:
gold_path = '/content/drive/MyDrive/DataStorm_Project/data/gold/'

In [16]:
# Loading cleaned data
df_trans = pd.read_csv(silver_path + 'clean_transactions.csv')

In [17]:
# Loading cleaned data
df_outlet = pd.read_csv(silver_path + 'clean_outlet_master.csv')

In [18]:
# Calculating Total Volume by store and month
# (Since there may be many bills in one store per month, they need to be added together)

monthly_sales = df_trans.groupby(['Outlet_ID', 'Year', 'Month'])['Volume_Liters'].sum().reset_index()

In [19]:
# Latent Potential (Base) Calculation - 95th Percentile Method
# The maximum average potential shown in the history of each store

def calculate_potential(x):
    return np.percentile(x, 95) # 95th Percentile

outlet_potential = monthly_sales.groupby('Outlet_ID')['Volume_Liters'].apply(calculate_potential).reset_index()
outlet_potential.rename(columns={'Volume_Liters': 'Base_Latent_Potential'}, inplace=True)

In [20]:
# 4. Merge this Potential with Outlet Master data
gold_df = pd.merge(df_outlet, outlet_potential, on='Outlet_ID', how='left')

In [21]:
# (If there are stores that never have sales, set their Potential to 0)
gold_df['Base_Latent_Potential'] = gold_df['Base_Latent_Potential'].fillna(0)

In [22]:
# Saving to the Gold Layer

gold_df.to_csv(gold_path + 'outlet_base_features.csv', index=False)

In [23]:
print("Base Latent Potential calculated and successfully transferred to the Gold Layer!")
print(gold_df[['Outlet_ID', 'Outlet_Type', 'Base_Latent_Potential']].head())

Base Latent Potential calculated and successfully transferred to the Gold Layer!
   Outlet_ID Outlet_Type  Base_Latent_Potential
0  OUT_00001      Grocry            1401.550605
1  OUT_00002       Hotel            1444.502824
2  OUT_00003    Pharmacy            1343.867804
3  OUT_00004    Pharmacy            1483.364791
4  OUT_00005       Kiosk            1150.363172


In [21]:
import pandas as pd
import requests
import numpy as np
from sklearn.neighbors import BallTree

In [26]:
bronze_path = '/content/drive/MyDrive/DataStorm_Project/data/bronze/'

In [2]:
print("Data collection for schools and bus stops across Sri Lanka begins...")

Data collection for schools and bus stops across Sri Lanka begins...


In [8]:
# Obtaining data related to the entire Sri Lanka at once
overpass_url = "https://overpass-api.de/api/interpreter"
overpass_query = """
[out:json][timeout:180];
(
  nwr["amenity"="school"](5.8, 79.5, 9.9, 82.0);
  nwr["highway"="bus_stop"](5.8, 79.5, 9.9, 82.0);
);
out center;
"""
# Sending a User-Agent is mandatory
headers = {'User-Agent': 'DataStorm_Participant_LK'}
response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers)
data = response.json()

In [13]:
# Extracting the obtained data
poi_data = []
for element in data.get('elements', []):
    lat = element.get('lat') or element.get('center', {}).get('lat')
    lon = element.get('lon') or element.get('center', {}).get('lon')
    tags = element.get('tags', {})

    poi_type = 'school' if tags.get('amenity') == 'school' else 'bus_stop'
    if lat and lon:
        poi_data.append([lat, lon, poi_type])

df_poi = pd.DataFrame(poi_data, columns=['lat', 'lon', 'type'])
df_schools = df_poi[df_poi['type'] == 'school'].copy()
df_bus_stops = df_poi[df_poi['type'] == 'bus_stop'].copy()


In [14]:
print(f"{len(df_schools)} schools from Sri Lanka and {len(df_bus_stops)} bus stops found!")

5029 schools from Sri Lanka and 3437 bus stops found!


In [15]:
print("\nPerforming Spatial Mapping for 20,000 shops...")


Performing Spatial Mapping for 20,000 shops...


In [18]:
# 2. Loading data from our 20,000 stores
bronze_path = '/content/drive/MyDrive/DataStorm_Project/data/bronze/'
gold_path = '/content/drive/MyDrive/DataStorm_Project/data/gold/'
df_coords = pd.read_csv(bronze_path + 'outlet_coordinates.csv')

In [19]:
# Converting latitude/longitude to radians for the BallTree algorithm
outlet_rad = np.deg2rad(df_coords[['Latitude', 'Longitude']])
school_rad = np.deg2rad(df_schools[['lat', 'lon']])
bus_rad = np.deg2rad(df_bus_stops[['lat', 'lon']])

In [23]:
# Creating Spatial Trees
school_tree = BallTree(school_rad, metric='haversine')
bus_tree = BallTree(bus_rad, metric='haversine')

In [24]:
# Calculating the distance of 1000 meters (1km) in radians (Earth's radius = 6371 km)
radius_rad = 1.0 / 6371.0


In [26]:
# Search for POIs for all 20,000 stores at once
df_coords['Schools_Within_1km'] = school_tree.query_radius(outlet_rad, r=radius_rad, count_only=True)
df_coords['Bus_Stops_Within_1km'] = bus_tree.query_radius(outlet_rad, r=radius_rad, count_only=True)

In [28]:
# Saving to the Gold Layer
df_coords.to_csv(gold_path + 'outlet_poi_features.csv', index=False)

In [29]:
print("Successful! External data for all 20,000 stores was retrieved and stored in the Gold Layer.")
print(df_coords.head())

Successful! External data for all 20,000 stores was retrieved and stored in the Gold Layer.
   Outlet_ID  Latitude  Longitude  Schools_Within_1km  Bus_Stops_Within_1km
0  OUT_00001  7.089846  79.979055                   2                     0
1  OUT_00002  7.000558  80.012422                   1                     0
2  OUT_00003  6.806170  79.854547                   0                     0
3  OUT_00004  6.703533  79.806919                   0                     0
4  OUT_00005  7.186878  79.869831                   1                     0


**Final Potential = Base Potential * (1 + POI Boost) * Seasonality Multiplier**

In [32]:
import pandas as pd
import numpy as np

# Folder Path

bronze_path = '/content/drive/MyDrive/DataStorm_Project/data/bronze/'
silver_path = '/content/drive/MyDrive/DataStorm_Project/data/silver/'
gold_path = '/content/drive/MyDrive/DataStorm_Project/data/gold/'

print("Predictions are being processed...")

# 1. Loading data

df_base = pd.read_csv(gold_path + 'outlet_base_features.csv')
df_poi = pd.read_csv(gold_path + 'outlet_poi_features.csv')
df_trans = pd.read_csv(silver_path + 'clean_transactions.csv')
df_season = pd.read_csv(bronze_path + 'distributor_seasonality_details.csv')

# 2. Obtaining the Distributor_ID related to the store (via Transactions)
outlet_dist = df_trans[['Outlet_ID', 'Distributor_ID']].drop_duplicates(subset=['Outlet_ID'])

# 3. Separate seasonality data for January (Month == 1)
jan_season = df_season[df_season['Month'] == 1].groupby('Distributor_ID')['Seasonality_Index'].last().reset_index()

# 4. Merging all data together

df_final = pd.merge(df_base, df_poi[['Outlet_ID', 'Schools_Within_1km', 'Bus_Stops_Within_1km']], on='Outlet_ID', how='left')
df_final = pd.merge(df_final, outlet_dist, on='Outlet_ID', how='left')
df_final = pd.merge(df_final, jan_season, on='Distributor_ID', how='left')

# Making NaN values ​​0

df_final.fillna({'Schools_Within_1km': 0, 'Bus_Stops_Within_1km': 0, 'Seasonality_Index': 'Moderate'}, inplace=True)

# 5. Implementing Causal Base Logic

def calculate_final_potential(row):
    base = row['Base_Latent_Potential']

    # Return 0 if base is less than 0 or empty
    if pd.isna(base) or base <= 0:
        return 0

   # Initial values
    poi_boost = 0
    s_mult = 1.00

   # POI Boost calculation
    schools = row['Schools_Within_1km'] if pd.notna(row['Schools_Within_1km']) else 0
    bus_stops = row['Bus_Stops_Within_1km'] if pd.notna(row['Bus_Stops_Within_1km']) else 0

    calculated_boost = (schools * 0.02) + (bus_stops * 0.01)
    poi_boost = min(calculated_boost, 0.10)

    # Seasonality Multiplier
    seasonality = row['Seasonality_Index']
    if seasonality == 'Favorable':
        s_mult = 1.05
    elif seasonality == 'Un-Favorable':
        s_mult = 0.95
    else:
        s_mult = 1.00

    # finall callculation
    final_potential = base * (1 + poi_boost) * s_mult
    return round(final_potential, 2)

df_final['Maximum_Monthly_Liters'] = df_final.apply(calculate_final_potential, axis=1)

# 6. Prepare the final CSV 2 columns
submission = df_final[['Outlet_ID', 'Maximum_Monthly_Liters']]

submission_file = '/content/drive/MyDrive/DataStorm_Project/Cylodea_predictions.csv'
submission.to_csv(submission_file, index=False)

print(f"Successful! Final answer file saved: {submission_file}")
print(submission.head())

Predictions are being processed...
Successful! Final answer file saved: /content/drive/MyDrive/DataStorm_Project/Cylodea_predictions.csv
   Outlet_ID  Maximum_Monthly_Liters
0  OUT_00001                 1457.61
1  OUT_00002                 1473.39
2  OUT_00003                 1343.87
3  OUT_00004                 1483.36
4  OUT_00005                 1173.37


In [28]:
# Loading Coordinates data
# df_coords = pd.read_csv(bronze_path + 'outlet_coordinates.csv')

In [44]:
# Test only 5 stores first (to avoid API blocking)
# df_sample = df_coords.head(5).copy()

In [5]:
# # def get_poi_counts(lat, lon, radius=1000):
# def get_poi_counts_updated(lat, lon, radius=1000):

#     """Find the number of schools and bus stops within 1000 meters of the given location"""
#     overpass_url = "http://overpass-api.de/api/interpreter"

#   # Use nwr (node, way, relation) and out center instead of node
#     # overpass_query = f"""
#     # [out:json];
#     # (
#     #   node["amenity"="school"](around:{radius},{lat},{lon});
#     #   node["highway"="bus_stop"](around:{radius},{lat},{lon});
#     # );
#     # out;
#     # """
#     overpass_query = f"""
#     [out:json];
#     (
#       nwr["amenity"="school"](around:{radius},{lat},{lon});
#       nwr["highway"="bus_stop"](around:{radius},{lat},{lon});
#     );
#     out center;
#     """
#     try:
#         response = requests.get(overpass_url, params={'data': overpass_query})


#         if response.status_code == 429:
#           print("API Limit reached, sleeping for 5 seconds....")
#           time.sleep(5)
#           return pd.Series([0, 0])

#         data = response.json()

#         schools = 0
#         bus_stops = 0

#         # Separate and calculate the results
#         for element in data.get('elements', []):
#             tags = element.get('tags', {})
#             if tags.get('amenity') == 'school':
#                 schools += 1
#             if tags.get('highway') == 'bus_stop':
#                 bus_stops += 1

#         return pd.Series([schools, bus_stops])

#     except Exception as e:
#         print(f"Error fetching data: {e}")
#         return pd.Series([0, 0])

# # print("Getting external data via API starts...")

# print("Data search begins via the Updated API...")

In [4]:
# # Executing the function related to our Sample (with an interval of 1 second)
# poi_results = []
# for index, row in df_sample.iterrows():
#     print(f"[{row['Outlet_ID']}] Searching for data ...")
#     # counts = get_poi_counts(row['Latitude'], row['Longitude'])
#     counts = get_poi_counts_updated(row['Latitude'], row['Longitude'])
#     poi_results.append(counts)
#     time.sleep(2)# Wait (2) seconds to avoid overloading the API server

In [41]:
# # Data collection
# df_sample[['Schools_Nearby', 'Bus_Stops_Nearby']] = pd.DataFrame(poi_results, index=df_sample.index)

In [3]:
# # print("\nData successfully retrieved!")
# print("\n New result:")
# print(df_sample)